In [17]:
import feedparser
import pandas as pd
from datetime import datetime

# Step 1: Define RSS feeds by ticker/topic
rss_feeds = {
    "AAPL": "https://finance.yahoo.com/rss/headline?s=AAPL",
    "TSLA": "https://finance.yahoo.com/rss/headline?s=TSLA",
    "MSFT": "https://finance.yahoo.com/rss/headline?s=MSFT",
    "General": "https://finance.yahoo.com/rss/topstories"
}

# Step 2: Scrape all feeds
records = []

for ticker, url in rss_feeds.items():
    feed = feedparser.parse(url)
    
    for entry in feed.entries:
        title = entry.get("title", None)
        published = entry.get("published", None)
        source = entry.get("source", {}).get("title", None)
        
        records.append({
            "ticker": ticker,
            "title": title,
            "published": published,
            "source": source
        })

# Step 3: Convert to DataFrame
df = pd.DataFrame(records)
print(df.shape)
print(df.head())

# Step 4: Save to CSV
df.to_csv("../data/raw/headlines_raw.csv", index=False)
print("Saved to data/raw/headlines_raw.csv")

(105, 4)
  ticker                                              title  \
0   AAPL  Prominent Tech Investor: ‘When We Were Buying ...   
1   AAPL  Apple shares slip as analysts offer mixed read...   
2   AAPL  Micron, Marvell, Super Micro, Coherent, Smucke...   
3   AAPL  Equities Markets Fall Intraday Amid Tech Sell-Off   
4   AAPL  Palantir vs. Apple: Which Stock Offers More Up...   

                         published source  
0  Tue, 09 Jun 2026 18:54:30 +0000    NaN  
1  Tue, 09 Jun 2026 18:27:00 +0000    NaN  
2  Tue, 09 Jun 2026 18:25:00 +0000    NaN  
3  Tue, 09 Jun 2026 18:17:06 +0000    NaN  
4  Tue, 09 Jun 2026 18:13:04 +0000    NaN  
Saved to data/raw/headlines_raw.csv


In [18]:
from transformers import pipeline
import pandas as pd

# Load FinBERT
finbert = pipeline("text-classification", model="ProsusAI/finbert")

# Load raw headlines
df = pd.read_csv("../data/raw/headlines_raw.csv")

# Run FinBERT on each headline
results = []
for title in df['title']:
    output = finbert([title])
    results.append({
        'sentiment_label': output[0]['label'],
        'sentiment_score': output[0]['score']
    })

results_df = pd.DataFrame(results)
df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

# Flag low confidence predictions
CONFIDENCE_THRESHOLD = 0.75
df['low_confidence'] = df['sentiment_score'] < CONFIDENCE_THRESHOLD

# Save labeled dataset
df.to_csv("../data/raw/headlines_labeled.csv", index=False)

print(df['sentiment_label'].value_counts())
print(f"\nLow confidence rows: {df['low_confidence'].sum()}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14411.94it/s]


sentiment_label
neutral     65
negative    23
positive    17
Name: count, dtype: int64

Low confidence rows: 18


In [19]:
import pandas as pd

# Load Financial PhraseBank from CSV
df_phrasebank = pd.read_csv("../data/raw/phrasebank.csv", 
                            header=None, 
                            names=['sentiment_label', 'title'],
                            encoding='latin-1')

df_phrasebank['source'] = 'financial_phrasebank'
df_phrasebank['low_confidence'] = False
df_phrasebank['sentiment_score'] = 1.0
df_phrasebank = df_phrasebank[['title', 'sentiment_label', 'sentiment_score', 'low_confidence', 'source']]

print(df_phrasebank['sentiment_label'].value_counts())
print(df_phrasebank.head())

# Load scraped headlines, exclude low confidence rows
df_rss = pd.read_csv("../data/raw/headlines_labeled.csv")
df_rss_filtered = df_rss[df_rss['low_confidence'] == False].copy()
df_rss_filtered['source'] = 'rss_finbert'
df_rss_filtered = df_rss_filtered[['title', 'sentiment_label', 'sentiment_score', 'low_confidence', 'source']]

# Merge both sources
df_merged = pd.concat([df_phrasebank, df_rss_filtered], ignore_index=True)

# Save
df_merged.to_csv("../data/processed/merged_dataset.csv", index=False)

print(f"\nFinancial PhraseBank rows: {len(df_phrasebank)}")
print(f"RSS rows after confidence filter: {len(df_rss_filtered)}")
print(f"Total merged rows: {len(df_merged)}")
print(f"\nLabel distribution:\n{df_merged['sentiment_label'].value_counts()}")

sentiment_label
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64
                                               title sentiment_label  \
0  According to Gran , the company has no plans t...         neutral   
1  Technopolis plans to develop in stages an area...         neutral   
2  The international electronic industry company ...        negative   
3  With the new production plant the company woul...        positive   
4  According to the company 's updated strategy f...        positive   

   sentiment_score  low_confidence                source  
0              1.0           False  financial_phrasebank  
1              1.0           False  financial_phrasebank  
2              1.0           False  financial_phrasebank  
3              1.0           False  financial_phrasebank  
4              1.0           False  financial_phrasebank  

Financial PhraseBank rows: 4846
RSS rows after confidence filter: 87
Total merged rows: 4933

Label distribution:
senti